**YAGA PROJECT**

*Fase 1. Creación DATASET*

**Objetivos:** 

**1.1.** Construir un motor capaz de extraer de forma automatizada los datos no estructurados de los reportes semanales que Uber manda a los conductores. 

**1.2** Desarrollar un agente autónomo de intercepción de red (Session Hijacking / Web Scraping) que capture el tráfico de datos crudos (JSON/XHR) interno de la plataforma, mapeando las coordenadas geográficas y métricas operativas de cada viaje.



**AGENTE DE DESCARGA PDF**

**Fase 1.1. Automatización de descarga documental**

-La clase `YagaPdfDownloader` utiliza `Playwright` para inicializar un contexto de navegación persistente (`uber_session`). Esto inyecta las *cookies* de sesión directamente, evadiendo los bloqueos de seguridad y el inicio de sesión manual.

-El bot se dirige al panel financiero y espera pasivamente la confirmación visual del operador.

-El script analiza las filas de la tabla (`<tr>`). Valida cruzadamente que el texto coincida con el mes objetivo y que exista un botón de descarga.

-Intercepta el flujo de descarga del navegador y guarda los archivos PDF directamente en una carpeta estructurada (`Uber_Reports/[Mes]_[Año]`), dejándola lista para la fase siguiente que es la auditoria_PDF. 


In [1]:
import asyncio
import os
import random
from playwright.async_api import async_playwright
import pdfplumber
import re
import glob
import json
import pandas as pd
from datetime import datetime
import nest_asyncio
from datetime import datetime, timezone, timedelta
import urllib.parse


**YagaPdfDownloader (Data Extraction & Ingestion)**

In [ ]:
class YagaPdfDownloader:
    def __init__(self, session_dir="./uber_session"):    # 1. Configuración inicio de sesion automatico carpeta "uber_sessio"
        self.session_dir = os.path.abspath(session_dir)
        self.base_url = "https://drivers.uber.com/earnings/statements"
        if not os.path.exists(self.session_dir):
            os.makedirs(self.session_dir)

    def _procesar_input(self, entrada: str):            
        try:
            partes = entrada.strip().lower().split()
            mes = partes[0]
            anio = partes[1]
            abreviatura = mes[:3]
            return mes, anio, abreviatura
        except IndexError:
            return None, None, None

    async def _iniciar_extraccion(self, page, mes_objetivo, anio_objetivo, mes_abreviado):  #3. La extracción crear la carpeta "Uber_Reports" de acuerdo al mes y año de extraccion
        carpeta_salida = f"Uber_Reports/{mes_objetivo.capitalize()}_{anio_objetivo}"
        os.makedirs(carpeta_salida, exist_ok=True)
        
        print(f"\n🚀 Iniciando descarga para {mes_objetivo.capitalize()} {anio_objetivo}...")
        descargas_exitosas = 0

        try:
            filas = await page.locator("tr").all()    # Captura todas las filas de la tabla de estados de cuenta

            for fila in filas:
                texto_fila = (await fila.inner_text()).lower()
                
                if mes_abreviado in texto_fila and "descargar pdf" in texto_fila:   # Validacion boton "Descargar PDF"
                    boton = fila.get_by_role("button", name="Descargar PDF")
                    
                    if await boton.is_visible():
                        await boton.scroll_into_view_if_needed()
                        
                        async with page.expect_download() as download_info:
                            await boton.click()                                       # Manej de la descarga
                        
                        download = await download_info.value
                        ruta_final = os.path.join(carpeta_salida, download.suggested_filename)
                        await download.save_as(ruta_final)
                        
                        print(f"✅ Guardado: {download.suggested_filename}")
                        descargas_exitosas += 1
                        
                        await asyncio.sleep(random.uniform(0.5, 1.5))
            
            print(f"\n ¡Misión cumplida! Se consolidaron {descargas_exitosas} documentos en '{carpeta_salida}'.")

        except Exception as e:
            print(f"❌ Error crítico en el pipeline de descarga: {e}")

    async def ejecutar(self):
        print("="*60)
        print("🤖 YAGA PDFDOWNLOADER")
        print("="*60)
        
        
        entrada = input("¿Qué mes y año vas a descargar? (ej: marzo 2025): ")      # Interfaz de Usuario   
        mes, anio, abreviatura = self._procesar_input(entrada)
        
        if not mes:
            print("❌ Formato incorrecto. Debes escribir el mes y el año separados por un espacio.")
            return

        async with async_playwright() as p:
            print("Cargando credenciales de sesión persistente...")
            context = await p.chromium.launch_persistent_context(
                self.session_dir,
                headless=False,
                args=["--no-sandbox", "--disable-dev-shm-usage"]
            )
            
            page = context.pages[0]
            await page.goto(self.base_url, wait_until="networkidle")
            
            print("\n" + "-"*60)
            print("🛠️ ACCIÓN REQUERIDA INTERVENCIÓN DEL USUARIO):")
            print(f"1. Selecciona en el navegador el Año '{anio}'.")
            print(f"2. Selecciona el Mes de '{mes.capitalize()}'.")
            print("3. Confirma visualmente que los botones 'Descargar PDF' están en pantalla.")
            print("-" * 60)
            
            input("PRESIONA [ENTER] AQUÍ CUANDO LA TABLA ESTÉ LISTA PARA EXTRACCIÓN...")   #Input del usuario para descargar automaticamente

            
            await self._iniciar_extraccion(page, mes, anio, abreviatura)  # Ejecución del núcleo de descarga

            print("Cierre seguro del navegador...")
            await asyncio.sleep(2)
            await context.close()


bot_descarga = YagaPdfDownloader()
await bot_descarga.ejecutar()

**Fase 1.1.2. AUDITORÍA DE DATOS (Motor Regex)**

**OBJETIVO** 
Consultar el reporte mensual de los PDF.



In [4]:
class UberReportExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.full_text = ""
        self.page1_text = ""
        self.texto_plano = "" 
        self.data = {
            "Conductor": "Roman Yair Ortega",
            "Semana_Periodo": "Not found",
            "Viajes_Totales": 0,
            "Horas_Conectado": 0.0,
            "Monto_Bruto_Generado": 0.0,
            "Propinas": 0.0,
            "Incentivos": 0.0,
            "Impuestos": 0.0
        }

    def load_pdf(self):
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for index, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean_text = text.replace("\u0000", "")
                        self.full_text += clean_text + "\n"
                        if index == 0:
                            self.page1_text = clean_text
            
            self.texto_plano = " ".join(self.full_text.split())                # Texto en crudo
            return True
        except Exception as e:
            print(f"❌ Error loading PDF: {e}")
            return False

    def parse_data(self):
        # 1. Fechas 
        match_date = re.search(r"(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})", self.page1_text)
        if match_date: 
            self.data["Semana_Periodo"] = f"Del {match_date.group(1)} al {match_date.group(2)}"
        else:
            match_date_alt = re.search(r"(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})", self.page1_text, re.IGNORECASE)
            if match_date_alt:
                self.data["Semana_Periodo"] = re.sub(r"\s+\d{1,2}\s+[ap]\.\s+m\.", "", match_date_alt.group(1)).strip()

        # 2. Viajes
        match_trips = re.search(r"viajes completados[^\d]*(\d+)", self.full_text, re.IGNORECASE)
        if match_trips: self.data["Viajes_Totales"] = int(match_trips.group(1))
            
        # 3. Tiempo 
        match_time = re.search(r"Tiempo de trabajo efectivo.*?(\d+)\s*h.*?(\d+)\s*m", self.texto_plano, re.IGNORECASE)
        if match_time:
            self.data["Horas_Conectado"] = round(int(match_time.group(1)) + (int(match_time.group(2)) / 60.0), 2)

        # Convertir a flotantes para análisis financiero
        def to_float(val_str):
            val_str = re.sub(r'[^\d\.,]', '', val_str)
            if not val_str: return 0.0
            if ',' in val_str and '.' in val_str:
                if val_str.rfind(',') > val_str.rfind('.'): 
                    val_str = val_str.replace('.', '').replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            elif ',' in val_str:
                if len(val_str) > 3 and val_str[-3] == ',':
                    val_str = val_str.replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            return float(val_str)

        # 4.  Ingresos
        match_gross = re.search(r"(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_gross: self.data["Monto_Bruto_Generado"] = to_float(match_gross.group(1))

        match_tips = re.search(r"(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_tips: self.data["Propinas"] = to_float(match_tips.group(1))

        # 6. Incentivos 
        match_incentives = re.search(r"(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_incentives: self.data["Incentivos"] = to_float(match_incentives.group(1))

        match_taxes = re.search(r"Impuestos\s*(?:-?\s*\$?|-)\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_taxes: self.data["Impuestos"] = -abs(to_float(match_taxes.group(1)))


class UberDatasetManager:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.all_reports_data = []
        self.pdf_files = []

    def find_pdfs(self):
        self.pdf_files = glob.glob(os.path.join(self.folder_path, "*.pdf"))
        print(f"🔍 Se encontraron {len(self.pdf_files)} reportes PDF en '{self.folder_path}'.")
        return len(self.pdf_files) > 0

    def process_all_reports(self):
        print("⚙️ Procesando y extrayendo datos en memoria...")
        for file_path in self.pdf_files:
            extractor = UberReportExtractor(file_path)
            if extractor.load_pdf():
                extractor.parse_data()
                self.all_reports_data.append(extractor.data)

    def get_dataframe(self):
        return pd.DataFrame(self.all_reports_data) if self.all_reports_data else None


# ==========================================
# EJECUCIÓN: MODO VISUALIZACIÓN EN MEMORIA
# ==========================================

print("="*60)
print("🤖 YAGA: Validador de Reportes Mensuales (Sin guardado en disco)")
print("="*60)

base_path = "Uber_Reports"
print(f"Directorio base detectado: {base_path}/")
nombre_mes = input("Ingresa el nombre de la carpeta del mes a analizar (Ej: Mayo_2025): ").strip()

ruta_input = os.path.join(base_path, nombre_mes)

if os.path.exists(ruta_input):
    dataset_maker = UberDatasetManager(ruta_input)

    if dataset_maker.find_pdfs():
        dataset_maker.process_all_reports()
        df_uber = dataset_maker.get_dataframe()
        
        # Filtramos las semanas vacías
        df_uber_limpio = df_uber[df_uber["Viajes_Totales"] > 0].copy()
        
        print(f"✅ Extracción finalizada. {len(df_uber_limpio)} semanas con actividad detectadas.")
        
        # Formato de Datos (Visualización)
        formato_moneda = "${:,.2f}"
        formato_horas = "{:.2f} hrs"
        formato_viajes = '{:.0f}'
        
        df_estilizado = df_uber_limpio.style.format({
            'Monto_Bruto_Generado': formato_moneda,
            'Viajes_Totales': formato_viajes,
            'Incentivos': formato_moneda,
            'Impuestos': formato_moneda,
            'Horas_Conectado': formato_horas,
            'Propinas': formato_moneda
        }).set_caption(f"Radiografía Financiera: {nombre_mes}")
        
        print("\n📊 TABLA DE RESULTADOS (Solo Lectura)")
        display(df_estilizado)
        
    else:
        print(f"⚠️ No se encontraron archivos PDF en: {ruta_input}")
else:
    print(f"❌ Error Crítico: La carpeta '{nombre_mes}' no existe dentro de '{base_path}'.")
    print(f"🔍 Verifica que la ruta '{ruta_input}' sea correcta.")



🤖 YAGA: Validador de Reportes Mensuales (Sin guardado en disco)
Directorio base detectado: Uber_Reports/


Ingresa el nombre de la carpeta del mes a analizar (Ej: Mayo_2025):  Diciembre_2024


🔍 Se encontraron 6 reportes PDF en 'Uber_Reports/Diciembre_2024'.
⚙️ Procesando y extrayendo datos en memoria...
✅ Extracción finalizada. 4 semanas con actividad detectadas.

📊 TABLA DE RESULTADOS (Solo Lectura)


,Conductor,Semana_Periodo,Viajes_Totales,Horas_Conectado,Monto_Bruto_Generado,Propinas,Incentivos,Impuestos
2,Roman Yair Ortega,Del 09/12/2024 al 16/12/2024,70,38.75 hrs,"$9,819.44",$80.00,$0.00,"$-1,033.72"
3,Roman Yair Ortega,Del 16/12/2024 al 23/12/2024,95,44.37 hrs,"$10,556.71",$35.00,$250.00,"$-1,229.47"
4,Roman Yair Ortega,Del 23/12/2024 al 30/12/2024,112,45.55 hrs,"$9,792.29",$70.00,$442.38,"$-1,309.57"
5,Roman Yair Ortega,Del 30/12/2024 al 06/01/2025,91,32.90 hrs,"$6,593.04",$110.00,$0.00,$-999.18


**FASE 1.1.3. REPORTE GLOBAL 2024-2025**

In [5]:
# ==========================================
# 1. MOTOR DE EXTRACCIÓN (Mantenemos tu extractor)
# ==========================================
class UberReportExtractor:
    def __init__(self, file_path):
        self.file_path = file_path
        self.full_text = ""
        self.page1_text = ""
        self.texto_plano = "" 
        self.data = {
            "Conductor": "Roman Yair Ortega",
            "Semana_Periodo": "Not found",
            "Viajes_Totales": 0,
            "Horas_Conectado": 0.0,
            "Monto_Bruto_Generado": 0.0,
            "Propinas": 0.0,
            "Incentivos": 0.0,
            "Impuestos": 0.0
        }

    def load_pdf(self):
        try:
            with pdfplumber.open(self.file_path) as pdf:
                for index, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        clean_text = text.replace("\u0000", "")
                        self.full_text += clean_text + "\n"
                        if index == 0:
                            self.page1_text = clean_text
            self.texto_plano = " ".join(self.full_text.split())
            return True
        except Exception as e:
            return False

    def parse_data(self):
        match_date = re.search(r"(\d{2}/\d{2}/\d{4}).*?(\d{2}/\d{2}/\d{4})", self.page1_text)
        if match_date: 
            self.data["Semana_Periodo"] = f"Del {match_date.group(1)} al {match_date.group(2)}"
        else:
            match_date_alt = re.search(r"(\d{1,2}\s+[a-z]{3}\s+\d{4}.*?-.*?\d{4})", self.page1_text, re.IGNORECASE)
            if match_date_alt:
                self.data["Semana_Periodo"] = re.sub(r"\s+\d{1,2}\s+[ap]\.\s+m\.", "", match_date_alt.group(1)).strip()

        match_trips = re.search(r"viajes completados[^\d]*(\d+)", self.full_text, re.IGNORECASE)
        if match_trips: self.data["Viajes_Totales"] = int(match_trips.group(1))
            
        match_time = re.search(r"Tiempo de trabajo efectivo.*?(\d+)\s*h.*?(\d+)\s*m", self.texto_plano, re.IGNORECASE)
        if match_time:
            self.data["Horas_Conectado"] = round(int(match_time.group(1)) + (int(match_time.group(2)) / 60.0), 2)

        def to_float(val_str):
            val_str = re.sub(r'[^\d\.,]', '', val_str)
            if not val_str: return 0.0
            if ',' in val_str and '.' in val_str:
                if val_str.rfind(',') > val_str.rfind('.'): 
                    val_str = val_str.replace('.', '').replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            elif ',' in val_str:
                if len(val_str) > 3 and val_str[-3] == ',':
                    val_str = val_str.replace(',', '.')
                else:
                    val_str = val_str.replace(',', '')
            return float(val_str)

        match_gross = re.search(r"(?:Importe bruto que has generado|Monto bruto generado)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_gross: self.data["Monto_Bruto_Generado"] = to_float(match_gross.group(1))

        match_tips = re.search(r"(?:Propinas dadas por los usuarios|Propinas otorgadas)\s+\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_tips: self.data["Propinas"] = to_float(match_tips.group(1))

        match_incentives = re.search(r"(?:Otras ganancias|Ganancias adicionales|promoción y fomento)\s*\$?\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_incentives: self.data["Incentivos"] = to_float(match_incentives.group(1))

        match_taxes = re.search(r"Impuestos\s*(?:-?\s*\$?|-)\s*([\d\.,]+[\.,]\d{2})", self.texto_plano, re.IGNORECASE)
        if match_taxes: self.data["Impuestos"] = -abs(to_float(match_taxes.group(1)))

# ==========================================
# 2. MANAGER GLOBAL ORQUESTADO (Extracción + Limpieza)
# ==========================================
class YagaGlobalOrchestrator:
    def __init__(self, root_path="Uber_Reports"):
        self.root_path = root_path
        self.all_data = []

    def ejecutar_pipeline_completo(self):
        print(f"🔍 [Fase 1] Escaneando directorios en {self.root_path}...")
        pdf_pattern = os.path.join(self.root_path, "**", "*.pdf")
        archivos_encontrados = glob.glob(pdf_pattern, recursive=True)
        
        if not archivos_encontrados:
            print("❌ No se encontraron archivos PDF.")
            return None

        print(f"📄 Se encontraron {len(archivos_encontrados)} reportes. Extrayendo datos...")
        for file_path in archivos_encontrados:
            extractor = UberReportExtractor(file_path)
            if extractor.load_pdf():
                extractor.parse_data()
                self.all_data.append(extractor.data)

        if not self.all_data:
            return None

        # Convertir a DataFrame y Filtrar Inactividad inmediatamente (En Memoria)
        df_bruto = pd.DataFrame(self.all_data)
        df_activos = df_bruto[df_bruto["Viajes_Totales"] > 0].copy()
        print(f"\n✔️ [Fase 2] Semanas activas identificadas: {len(df_activos)}")

        print("⚙️ [Fase 3] Iniciando reestructuración cronológica vectorizada...")
        
        # Extraer fechas y convertir a Datetime
        df_activos['rango_fechas'] = df_activos['Semana_Periodo']
        fecha_str = df_activos['Semana_Periodo'].str.extract(r'(\d{2}/\d{2}/\d{4})')[0]
        df_activos['fecha_dt'] = pd.to_datetime(fecha_str, format='%d/%m/%Y', errors='coerce')
        
        df_activos = df_activos.dropna(subset=['fecha_dt']).copy()

        # Ingeniería de Características (Feature Engineering Temporal)
        df_activos['anio'] = df_activos['fecha_dt'].dt.year.astype(int)
        meses_es = {
            1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
            5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
            9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
        }
        df_activos['mes'] = df_activos['fecha_dt'].dt.month.map(meses_es)

        # Ordenamiento Cronológico
        df_organizado = df_activos.sort_values(by='fecha_dt').reset_index(drop=True)
        fecha_origen = df_organizado['fecha_dt'].min()
        
        # Cálculo de semanas
        df_organizado['num_semana'] = ((df_organizado['fecha_dt'] - fecha_origen).dt.days // 7) + 1
        df_organizado['semana'] = "Semana " + df_organizado['num_semana'].astype(str)
        df_organizado['identificador_periodo'] = df_organizado['mes'] + " - " + df_organizado['semana']

        # Limpieza financiera (Data Cleansing)
        columnas_financieras = ['Monto_Bruto_Generado', 'Propinas', 'Incentivos', 'Impuestos']
        for col in columnas_financieras:
            if col in df_organizado.columns:
                df_organizado[col] = df_organizado[col].fillna(0.0)

        # Reordenamiento Final
        df_organizado.drop(columns=['Semana_Periodo', 'fecha_dt', 'num_semana'], inplace=True, errors='ignore')
        nuevas_cols = ['anio', 'mes', 'semana', 'identificador_periodo', 'rango_fechas']
        columnas_restantes = [c for c in df_organizado.columns if c not in nuevas_cols]
        
        df_final = df_organizado[nuevas_cols + columnas_restantes]
        
        return df_final, fecha_origen

# ==========================================
# 3. EJECUCIÓN DEL PIPELINE Y REPORTE
# ==========================================
orquestador = YagaGlobalOrchestrator()
resultado = orquestador.ejecutar_pipeline_completo()

if resultado is not None:
    df_yaga_historico, fecha_origen = resultado
    
    # EXPORTACIÓN ÚNICA
    output_name = 'YAGA_DataSet_Maestro_Organizado.csv'
    df_yaga_historico.to_csv(output_name, index=False)
    
    print("\n" + "="*50)
    print("🚀 ¡ÉXITO! YAGA DATASET GENERADO")
    print("="*50)
    print(f"📅 Fecha de Origen (Semana 1): {fecha_origen.strftime('%d/%b/%Y')}")
    print(f"💾 Archivo Maestro depositado: {output_name}")
    
    # --- RESUMEN ANUAL ---
    print("\n📈 GLOBAL REPORT HISTÓRICO")
    print("-" * 50)
    total_viajes = int(df_yaga_historico['Viajes_Totales'].sum())
    ingreso_bruto = df_yaga_historico['Monto_Bruto_Generado'].sum()
    print(f"Total Viajes Históricos: {total_viajes:,}")
    print(f"Ingreso Bruto Total: ${ingreso_bruto:,.2f}") 
    print("-" * 50)
    
    # --- RENDERIZADO VISUAL ---
    formato_moneda = "${:,.2f}"
    formato_horas = "{:.2f} hrs"
    formato_viajes = "{:.0f}"  
    
    diccionario_formatos = {
        'Monto_Bruto_Generado': formato_moneda,
        'Propinas': formato_moneda,
        'Incentivos': formato_moneda,
        'Impuestos': formato_moneda,
        'Viajes_Totales': formato_viajes,
        'Horas_Conectado': formato_horas
    }
    
    # Vista previa de los primeros 15 registros cronológicos
    df_renderizado = df_yaga_historico.head(15).style.format(diccionario_formatos).set_caption("Vista Previa del Dataset Maestro")
    
    print("\n✅ VISTA PREVIA ORGANIZADA CRONOLÓGICAMENTE:")
    display(df_renderizado)

🔍 [Fase 1] Escaneando directorios en Uber_Reports...
📄 Se encontraron 138 reportes. Extrayendo datos...

✔️ [Fase 2] Semanas activas identificadas: 61
⚙️ [Fase 3] Iniciando reestructuración cronológica vectorizada...

🚀 ¡ÉXITO! YAGA DATASET GENERADO
📅 Fecha de Origen (Semana 1): 17/Jun/2024
💾 Archivo Maestro depositado: YAGA_DataSet_Maestro_Organizado.csv

📈 GLOBAL REPORT HISTÓRICO
--------------------------------------------------
Total Viajes Históricos: 4,097
Ingreso Bruto Total: $299,879.50
--------------------------------------------------

✅ VISTA PREVIA ORGANIZADA CRONOLÓGICAMENTE:


,anio,mes,semana,identificador_periodo,rango_fechas,Conductor,Viajes_Totales,Horas_Conectado,Monto_Bruto_Generado,Propinas,Incentivos,Impuestos
0,2024,Junio,Semana 1,Junio - Semana 1,Del 17/06/2024 al 24/06/2024,Roman Yair Ortega,15,10.07 hrs,"$1,815.54",$0.00,$32.41,$-299.69
1,2024,Junio,Semana 2,Junio - Semana 2,Del 24/06/2024 al 01/07/2024,Roman Yair Ortega,23,12.90 hrs,"$2,693.58",$20.00,$0.00,$-278.47
2,2024,Junio,Semana 2,Junio - Semana 2,Del 24/06/2024 al 01/07/2024,Roman Yair Ortega,23,12.90 hrs,"$2,693.58",$20.00,$0.00,$-278.47
3,2024,Julio,Semana 3,Julio - Semana 3,Del 01/07/2024 al 08/07/2024,Roman Yair Ortega,18,8.55 hrs,"$1,512.86",$20.00,$3.96,$-232.67
4,2024,Diciembre,Semana 26,Diciembre - Semana 26,Del 09/12/2024 al 16/12/2024,Roman Yair Ortega,70,38.75 hrs,"$9,819.44",$80.00,$0.00,"$-1,033.72"
5,2024,Diciembre,Semana 27,Diciembre - Semana 27,Del 16/12/2024 al 23/12/2024,Roman Yair Ortega,95,44.37 hrs,"$10,556.71",$35.00,$250.00,"$-1,229.47"
6,2024,Diciembre,Semana 28,Diciembre - Semana 28,Del 23/12/2024 al 30/12/2024,Roman Yair Ortega,112,45.55 hrs,"$9,792.29",$70.00,$442.38,"$-1,309.57"
7,2024,Diciembre,Semana 29,Diciembre - Semana 29,Del 30/12/2024 al 06/01/2025,Roman Yair Ortega,91,32.90 hrs,"$6,593.04",$110.00,$0.00,$-999.18
8,2024,Diciembre,Semana 29,Diciembre - Semana 29,Del 30/12/2024 al 06/01/2025,Roman Yair Ortega,91,32.90 hrs,"$6,593.04",$110.00,$0.00,$-999.18
9,2025,Enero,Semana 30,Enero - Semana 30,Del 06/01/2025 al 13/01/2025,Roman Yair Ortega,103,38.15 hrs,"$7,431.44",$220.00,$0.00,"$-1,088.13"


**Fase 1.2. BOOT EXTRACTOR.JSON PAGINA UBER**

Intercepcion de traficos de datos

Este módulo extrae la **Realidad Logística**, capturando coordenadas GPS, distancias y tiempos exactos por cada viaje realizado.

 > **⚠️ Nota de Arquitectura (Seguridad):** > Este módulo implementa un modelo **Human-in-the-Loop**. Debido a los protocolos de seguridad y anti-scraping de la plataforma, el bot requiere que el usuario realice la navegación inicial. Esto garantiza la persistencia de la sesión y evita el bloqueo de la cuenta por comportamiento automatizado predecible.

In [2]:
class YagaNetworkScraperTrimestral:
    def __init__(self, vault_path="Extraction_trafic_data_Uber", session_dir="uber_session"):
        self.vault_path = vault_path
        self.session_file = os.path.join(session_dir, "gs_session.json")
        self.buffer_actividades = {}
        
        os.makedirs(self.vault_path, exist_ok=True)
        os.makedirs(session_dir, exist_ok=True)

    async def _interceptar_todo(self, response):
        """Escucha silenciosa de la red (Modo Aspirador)."""
        if "getWebActivityFeed" in response.url and response.status == 200:
            try:
                datos_json = await response.json()
                actividades = datos_json.get('data', {}).get('activities', [])
                
                nuevos = 0
                for v in actividades:
                    uuid = v.get('uuid')
                    if uuid and uuid not in self.buffer_actividades:
                        self.buffer_actividades[uuid] = v
                        nuevos += 1
                
                if nuevos > 0:
                    print(f"📡 [Tráfico YAGA] Paquetes capturados. Total en memoria RAM: {len(self.buffer_actividades)}", end="\r")
            except Exception:
                pass

    def _procesar_captura(self, q_target, a_target, modo_prueba=False):
        """Descarga el buffer completo a la bóveda sin discriminar fechas."""
        print(f"\n\n⚖️ Consolidando Bóveda para la sesión Q{q_target}/{a_target}...")
        
        viajes = list(self.buffer_actividades.values())
        
        print("-" * 60)
        if viajes:
            # Ahora guardamos un solo archivo por sesión de extracción
            json_file = os.path.join(self.vault_path, f"viajes_crudos_{a_target}_Q{q_target}_detallado.json")

            if modo_prueba:
                print(f"🧪 PRUEBA -> Detectados {len(viajes)} paquetes (No guardados).")
            else:
                with open(json_file, "w", encoding="utf-8") as f:
                    json.dump(viajes, f, ensure_ascii=False, indent=4)
                print(f"🏆 EXTRACCIÓN EXITOSA -> {len(viajes)} paquetes guardados en: {os.path.basename(json_file)}")
        else:
            print("⚠️ AVISO CRÍTICO: No se detectó ningún paquete en la red. Verifica que la página cargó correctamente.")
        print("-" * 60)

    async def ejecutar(self, modo_prueba=False):
        self.buffer_actividades = {}
        
        print("\n" + "="*60)
        titulo = "🤖 YAGA CORE: EXTRACCIÓN DE RED (ASPIRADOR PURO)" + (" (🧪 PRUEBA)" if modo_prueba else " (💾 PRODUCCIÓN)")
        print(titulo)
        print("="*60)
        
        entrada = await asyncio.to_thread(input, "📅 Ingresa Trimestre y Año a extraer (ej. 2/2024): ")
        try:
            limpio = entrada.upper().replace('Q', '').replace('T', '').strip()
            q_target, a_target = map(int, limpio.split('/'))
        except Exception:
            print("❌ Formato inválido. Usa un trimestre del 1 al 4 (ej. 2/2024).")
            return

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=False)
            context = await browser.new_context(
                storage_state=self.session_file if os.path.exists(self.session_file) else None
            )
            page = await context.new_page()
            
            page.on("response", self._interceptar_todo)

            await page.goto("https://drivers.uber.com/earnings/activities", wait_until="networkidle")
            
            print(f"🌐 NAVEGADOR LISTO. Usa el calendario de Uber para ir a las fechas de Q{q_target} {a_target}.")
            print("Haz scroll hacia abajo para cargar todos los viajes posibles.")

            await asyncio.to_thread(input, "\n⏳ Presiona [ENTER] aquí cuando el contador de paquetes en RAM deje de subir...")

            self._procesar_captura(q_target, a_target, modo_prueba=modo_prueba)

            if not modo_prueba:
                await context.storage_state(path=self.session_file)
            await browser.close()

# ==========================================
# EJECUCIÓN 
# ==========================================
bot_trimestral = YagaNetworkScraperTrimestral()
await bot_trimestral.ejecutar(modo_prueba=False)


🤖 YAGA CORE: EXTRACCIÓN DE RED (ASPIRADOR PURO) (💾 PRODUCCIÓN)


📅 Ingresa Trimestre y Año a extraer (ej. 2/2024):  02/2025


TimeoutError: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://drivers.uber.com/earnings/activities", waiting until "networkidle"


**LIMPIEZA ARCHIVO JSON**

In [3]:
class YagaGlobalRefiner:
    def __init__(self, folder_path="Extraction_trafic_data_Uber"):
        self.folder_path = folder_path
        self.master_data = []

    @staticmethod
    def _parse_money(text: str) -> float:
        if not text: return 0.0
        cleaned = re.sub(r'[^\d\.,]', '', str(text))
        if ',' in cleaned and '.' in cleaned:
            cleaned = cleaned.replace('.', '').replace(',', '.') if cleaned.rfind(',') > cleaned.rfind('.') else cleaned.replace(',', '')
        elif ',' in cleaned:
            cleaned = cleaned.replace(',', '.') if len(cleaned) - cleaned.rfind(',') == 3 else cleaned.replace(',', '')
        try: return float(cleaned)
        except: return 0.0

    @staticmethod
    def _parse_duration(text: str) -> float:
        if not text: return 0.0
        m = re.search(r'(\d+)\s*min', text)
        s = re.search(r'(\d+)\s*seg', text)
        total = int(m.group(1)) if m else 0
        total += (int(s.group(1)) / 60.0) if s else 0
        return round(total, 2)

    def procesar_boveda_completa(self):
        patron = os.path.join(self.folder_path, "*_detallado.json")
        archivos = glob.glob(patron)
        archivos.sort()
        
        if not archivos:
            print("❌ No se encontraron archivos para procesar.")
            return None

        print(f"⚙️ Iniciando refinamiento global de {len(archivos)} bloques de datos...")

        for ruta in archivos:
            with open(ruta, "r", encoding="utf-8") as f:
                raw_data = json.load(f)

            count_limpios = 0
            for item in raw_data:
                # 1. Tiempo (Ajuste UTC-10)
                ts_val = item.get('recognizedAt') or (item.get('dropOffTime') / 1000 if item.get('dropOffTime') else 0)
                if not ts_val: continue
                ts_uber = datetime.fromtimestamp(int(ts_val), tz=timezone.utc) - timedelta(hours=10)

                # 2. Finanzas (Filtro de Justicia)
                monto_bruto = self._parse_money(item.get('formattedTotal', ''))
                if monto_bruto <= 0: continue # Ignorar "viajes fantasma" o cancelados sin cobro

                # 3. Metadatos
                meta = item.get('tripMetaData') or {}
                duracion = self._parse_duration(meta.get('formattedDuration', ''))
                distancia_km = self._parse_money(meta.get('formattedDistance', '0'))

                self.master_data.append({
                    'trip_id': item.get('uuid'),
                    'fecha_local': ts_uber.strftime('%Y-%m-%d %H:%M:%S'),
                    'anio': ts_uber.year,
                    'mes': ts_uber.strftime('%m'),
                    'monto_bruto': monto_bruto,
                    'duracion_min': duracion,
                    'distancia_km': distancia_km,
                    'mxn_por_minuto': round(monto_bruto / duracion, 2) if duracion > 0 else 0,
                    'estado': item.get('status'),
                    'origen': meta.get('pickupAddress'),
                    'destino': meta.get('dropOffAddress')
                })
                count_limpios += 1
            print(f"✔️ {os.path.basename(ruta)}: {count_limpios} viajes rentables extraídos.")

        # Generación de DataFrame Maestro
        df_final = pd.DataFrame(self.master_data)
        df_final = df_final.drop_duplicates(subset=['trip_id']).sort_values(by='fecha_local')
        return df_final

# ==========================================
# EJECUCIÓN DEL REFINAMIENTO GLOBAL
# ==========================================
print("="*60)
print("💎 YAGA CORE: GENERACIÓN DEL DATASET OPERATIVO MAESTRO")
print("="*60)

refiner = YagaGlobalRefiner()
df_yaga_operativo = refiner.procesar_boveda_completa()

if df_yaga_operativo is not None:
    output_name = "YAGA_DataSet_Operativo_Viajes_2024_2026.csv"
    df_yaga_operativo.to_csv(output_name, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*60)
    print(f"✅ ¡ÉXITO! DATASET MAESTRO GENERADO.")
    print(f"💾 Guardado como: {output_name}")
    print(f"🚗 Total de viajes únicos consolidados: {len(df_yaga_operativo):,}")
    print(f"💰 Suma monetaria bruta total: ${df_yaga_operativo['monto_bruto'].sum():,.2f}")
    print("-" * 60)
    
    # Vista previa
    display(df_yaga_operativo[['fecha_local', 'monto_bruto', 'distancia_km', 'mxn_por_minuto']].head())

💎 YAGA CORE: GENERACIÓN DEL DATASET OPERATIVO MAESTRO
⚙️ Iniciando refinamiento global de 15 bloques de datos...
✔️ viajes_2024_06_detallado.json: 40 viajes rentables extraídos.
✔️ viajes_2024_07_detallado.json: 23 viajes rentables extraídos.
✔️ viajes_2024_08_detallado.json: 1 viajes rentables extraídos.
✔️ viajes_2024_12_detallado.json: 279 viajes rentables extraídos.
✔️ viajes_2025_01_detallado.json: 521 viajes rentables extraídos.
✔️ viajes_2025_02_detallado.json: 340 viajes rentables extraídos.
✔️ viajes_2025_03_detallado.json: 393 viajes rentables extraídos.
✔️ viajes_crudos_2024_Q2_detallado.json: 40 viajes rentables extraídos.
✔️ viajes_crudos_2024_Q3_detallado.json: 24 viajes rentables extraídos.
✔️ viajes_crudos_2024_Q4_detallado.json: 372 viajes rentables extraídos.
✔️ viajes_crudos_2025_Q1_detallado.json: 1254 viajes rentables extraídos.
✔️ viajes_crudos_2025_Q2_detallado.json: 931 viajes rentables extraídos.
✔️ viajes_crudos_2025_Q3_detallado.json: 519 viajes rentables ext

,fecha_local,monto_bruto,distancia_km,mxn_por_minuto
15,2024-06-22 14:00:00,100.00,0.00,0.00
14,2024-06-22 14:40:25,137.55,18.75,3.77
13,2024-06-22 15:21:42,146.47,26.43,3.54
12,2024-06-22 16:21:45,90.52,6.51,5.26
11,2024-06-22 16:45:58,189.40,30.02,3.27


In [6]:
"""
YAGA PROJECT — Data Fusion Engine v8.0
Módulo: YagaDataFusionEngine
Combina la lógica compacta del v7.1 con el formatting profesional del XLSX.
"""
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Paleta ────────────────────────────────────────────────────────────────────
C = {
    'white':        "FFFFFFFF",
    'gris_claro':   "FFF5F5F5",
    'borde':        "FFD0D0D0",
    'header':       "FF2D2D2D",
    'purple_dark':  "FF3C3489",
    'purple_light': "FFEEEDFE",
    'teal_dark':    "FF0F6E56",
    'teal_light':   "FFE1F5EE",
    'coral_dark':   "FF993C1D",
    'coral_light':  "FFFAECE7",
    'amber_dark':   "FF854F0B",
    'amber_light':  "FFFAEEDA",
    'red_fill':     "FFFCEBEB",
    'green_fill':   "FFEAF3DE",
}

MESES_MAP = {
    '01':'Enero',   '02':'Febrero', '03':'Marzo',     '04':'Abril',
    '05':'Mayo',    '06':'Junio',   '07':'Julio',      '08':'Agosto',
    '09':'Septiembre', '10':'Octubre', '11':'Noviembre', '12':'Diciembre'
}
MESES_ORD = {v: int(k) for k, v in MESES_MAP.items()}


# ── Helpers de formato ────────────────────────────────────────────────────────
def _borde():
    s = Side(style='thin', color=C['borde'])
    return Border(top=s, bottom=s, left=s, right=s)

def _header_row(ws, headers, row=1, bg=C['header']):
    """Escribe una fila de cabecera con estilo."""
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=row, column=i, value=h)
        cell.font      = Font(name='Arial', bold=True, size=10, color=C['white'])
        cell.fill      = PatternFill('solid', fgColor=bg)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = _borde()
    ws.row_dimensions[row].height = 26

def _data_row(ws, row, values, bg=C['white'], fmt_map=None, left_cols=None):
    """Escribe una fila de datos con zebra striping opcional."""
    for i, v in enumerate(values, 1):
        cell = ws.cell(row=row, column=i, value=v)
        cell.font      = Font(name='Arial', size=10)
        cell.fill      = PatternFill('solid', fgColor=bg)
        cell.alignment = Alignment(
            horizontal='left' if (left_cols and i in left_cols) else 'center',
            vertical='center'
        )
        cell.border = _borde()
        if fmt_map and (i - 1) in fmt_map:
            cell.number_format = fmt_map[i - 1]

def _total_row(ws, row, sum_cols, n_cols, bg, fmt_map=None):
    """Fila de totales con fondo oscuro."""
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col)
        cell.fill   = PatternFill('solid', fgColor=bg)
        cell.border = _borde()
    ws.cell(row=row, column=1, value='TOTAL').font = Font(
        name='Arial', bold=True, size=10, color=C['white'])
    ws.cell(row=row, column=1).fill = PatternFill('solid', fgColor=bg)

    for col_idx, col_letter, nf in sum_cols:
        first_data = ws.cell(row=2, column=col_idx).row   # always row 2 for data start
        # use passed first_row arg instead
        c = ws.cell(row=row, column=col_idx)
        c.font           = Font(name='Arial', bold=True, size=10, color=C['white'])
        c.fill           = PatternFill('solid', fgColor=bg)
        c.border         = _borde()
        c.number_format  = nf

def _title_block(ws, title, subtitle, dark, light, last_col='M'):
    ws.merge_cells(f'A1:{last_col}1')
    t = ws['A1']
    t.value     = title
    t.font      = Font(name='Arial', bold=True, size=13, color=C['white'])
    t.fill      = PatternFill('solid', fgColor=dark)
    t.alignment = Alignment(horizontal='left', vertical='center')
    ws.row_dimensions[1].height = 30

    ws.merge_cells(f'A2:{last_col}2')
    s = ws['A2']
    s.value     = subtitle
    s.font      = Font(name='Arial', size=10, italic=True, color=dark)
    s.fill      = PatternFill('solid', fgColor=light)
    s.alignment = Alignment(horizontal='left', vertical='center')
    ws.row_dimensions[2].height = 18

def _col_widths(ws, widths):
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w


# ═══════════════════════════════════════════════════════════════════════════════
class YagaDataFusionEngine:
    """
    Motor de fusión YAGA v8.0
    - Lógica compacta (v7.1): un único método ejecutar_y_exportar()
    - Formatting profesional: títulos, zebra, totales, color-coding de deltas
    
    Parámetros
    ----------
    json_file  : CSV de tráfico de red (viajes individuales)
    pdf_file   : CSV de reportes PDF (semanas)
    vault_path : carpeta con *_alta_precision.csv (fallback si json_file no existe)
    """

    def __init__(self,
                 json_file:  str = "YAGA_DataSet_Operativo_ViajesJSON_2024_2026.csv",
                 pdf_file:   str = "YAGA_DataSet_PDF_Maestro_Organizado.csv",
                 vault_path: str = "Extraction_trafic_data_Uber"):
        self.json_file  = json_file
        self.pdf_file   = pdf_file
        self.vault_path = vault_path

    # ── Carga ─────────────────────────────────────────────────────────────────
    def _cargar_json(self) -> pd.DataFrame:
        if os.path.exists(self.json_file):
            df = pd.read_csv(self.json_file)
        else:
            archivos = glob.glob(os.path.join(self.vault_path, "*_alta_precision.csv"))
            if not archivos:
                raise FileNotFoundError(f"❌ No se encontró {self.json_file} ni archivos en {self.vault_path}")
            print(f"📦 Construyendo desde {len(archivos)} archivos en bóveda...")
            df = pd.concat([pd.read_csv(f) for f in archivos], ignore_index=True)

        df['fecha_local'] = pd.to_datetime(df['fecha_local'])
        df = df.drop_duplicates(subset=['trip_id']).sort_values('fecha_local').reset_index(drop=True)
        df['mes_texto'] = df['mes'].astype(str).str.zfill(2).map(MESES_MAP)
        return df

    def _cargar_pdf(self) -> pd.DataFrame:
        if not os.path.exists(self.pdf_file):
            raise FileNotFoundError(f"❌ No se encontró {self.pdf_file}")
        df = pd.read_csv(self.pdf_file)
        # HOTFIX: purgar semanas duplicadas antes de sumar
        return df.drop_duplicates(subset=['anio', 'mes', 'semana'])

    # ── Hoja 1: Viajes RED ────────────────────────────────────────────────────
    def _sheet_viajes(self, wb, df_json):
        ws = wb.create_sheet("Viajes_RED")
        _title_block(ws, "YAGA — Viajes Individuales (Tráfico de Red)",
                     f"Fuente: JSON/XHR  ·  {len(df_json):,} viajes únicos",
                     C['purple_dark'], C['purple_light'], last_col='L')

        headers = ['#', 'Trip ID', 'Fecha y Hora', 'Año', 'Mes', 'Estado',
                   'Monto Bruto (MXN)', 'Duración (min)', 'Distancia (km)',
                   'MXN/min', 'Origen', 'Destino']
        _header_row(ws, headers, row=4, bg=C['purple_dark'])

        fmt = {6: '$#,##0.00', 7: '0.00', 8: '0.00', 9: '0.000'}
        for i, row in df_json.iterrows():
            r   = i + 5
            bg  = C['gris_claro'] if i % 2 == 0 else C['white']
            vals = [
                i + 1, row['trip_id'],
                row['fecha_local'].strftime('%Y-%m-%d %H:%M'),
                row['anio'], row['mes_texto'], row.get('estado', ''),
                row['monto_bruto'], row['duracion_min'], row['distancia_km'],
                row['mxn_por_minuto'], row.get('origen', ''), row.get('destino', '')
            ]
            _data_row(ws, r, vals, bg=bg, fmt_map=fmt, left_cols={2, 11, 12})

        # Fila de totales
        tr = len(df_json) + 5
        for col in range(1, 13):
            ws.cell(tr, col).fill   = PatternFill('solid', fgColor=C['purple_dark'])
            ws.cell(tr, col).border = _borde()
        ws.cell(tr, 1, 'TOTAL').font = Font(name='Arial', bold=True, size=10, color=C['white'])
        ws.cell(tr, 1).fill = PatternFill('solid', fgColor=C['purple_dark'])
        for ci, cl, nf in [(7,'G','$#,##0.00'), (8,'H','0.00'), (9,'I','0.00')]:
            c = ws.cell(tr, ci, f'=SUM({cl}5:{cl}{tr-1})')
            c.font   = Font(name='Arial', bold=True, size=10, color=C['white'])
            c.fill   = PatternFill('solid', fgColor=C['purple_dark'])
            c.border = _borde()
            c.number_format = nf

        ws.freeze_panes = 'A5'
        _col_widths(ws, [5, 38, 18, 6, 12, 12, 18, 14, 14, 10, 45, 45])

    # ── Hoja 2: Reporte PDF ───────────────────────────────────────────────────
    def _sheet_pdf(self, wb, df_pdf):
        ws = wb.create_sheet("Reporte_PDF")
        _title_block(ws, "YAGA — Reporte Semanal (Extracción PDF)",
                     f"Fuente: Reportes PDF Uber  ·  {len(df_pdf):,} semanas",
                     C['teal_dark'], C['teal_light'], last_col='L')

        headers = ['Año', 'Mes', 'Semana', 'Periodo', 'Rango de Fechas', 'Conductor',
                   'Viajes', 'Horas Conectado', 'Monto Bruto (MXN)',
                   'Propinas (MXN)', 'Incentivos (MXN)', 'Impuestos (MXN)']
        _header_row(ws, headers, row=4, bg=C['teal_dark'])

        fmt = {7: '0.00', 8: '$#,##0.00', 9: '$#,##0.00', 10: '$#,##0.00', 11: '$#,##0.00'}
        for i, row in df_pdf.iterrows():
            r  = i + 5
            bg = C['gris_claro'] if i % 2 == 0 else C['white']
            vals = [
                row['anio'], row['mes'], row['semana'],
                row.get('identificador_periodo', ''), row.get('rango_fechas', ''),
                row.get('Conductor', ''),
                row['Viajes_Totales'], row['Horas_Conectado'], row['Monto_Bruto_Generado'],
                row['Propinas'], row['Incentivos'], row['Impuestos']
            ]
            _data_row(ws, r, vals, bg=bg, fmt_map=fmt, left_cols={4, 5, 6})

        # Fila de totales
        tr = len(df_pdf) + 5
        for col in range(1, 13):
            ws.cell(tr, col).fill   = PatternFill('solid', fgColor=C['teal_dark'])
            ws.cell(tr, col).border = _borde()
        ws.cell(tr, 1, 'TOTAL').font = Font(name='Arial', bold=True, size=10, color=C['white'])
        ws.cell(tr, 1).fill = PatternFill('solid', fgColor=C['teal_dark'])
        for ci, cl, nf in [(7,'G','0'), (8,'H','0.00'), (9,'I','$#,##0.00'),
                           (10,'J','$#,##0.00'), (11,'K','$#,##0.00'), (12,'L','$#,##0.00')]:
            c = ws.cell(tr, ci, f'=SUM({cl}5:{cl}{tr-1})')
            c.font   = Font(name='Arial', bold=True, size=10, color=C['white'])
            c.fill   = PatternFill('solid', fgColor=C['teal_dark'])
            c.border = _borde()
            c.number_format = nf

        ws.freeze_panes = 'A5'
        _col_widths(ws, [6, 14, 12, 24, 28, 20, 8, 15, 18, 16, 16, 16])

    # ── Hoja 3: Auditoría PDF vs RED ──────────────────────────────────────────
    def _sheet_auditoria(self, wb, df_audit):
        ws = wb.create_sheet("Auditoria_PDF_vs_RED")
        _title_block(ws, "YAGA — Auditoría: PDF vs Tráfico de Red",
                     "Discrepancia mensual  ·  PDF (Uber oficial) vs JSON/XHR (captura de red)",
                     C['coral_dark'], C['coral_light'], last_col='Q')

        # Sub-cabeceras de grupo (fila 4)
        grupos = [
            ('A4:B4', '',                        C['header']),
            ('C4:H4', 'VERDAD CONTABLE — PDF',    C['teal_dark']),
            ('I4:L4', 'REALIDAD LOGÍSTICA — RED', C['purple_dark']),
            ('M4:Q4', 'DISCREPANCIA',             C['coral_dark']),
        ]
        for rng, label, color in grupos:
            ws.merge_cells(rng)
            c = ws[rng.split(':')[0]]
            c.value     = label
            c.font      = Font(name='Arial', bold=True, size=9, color=C['white'])
            c.fill      = PatternFill('solid', fgColor=color)
            c.alignment = Alignment(horizontal='center', vertical='center')
        ws.row_dimensions[4].height = 20

        headers = [
            'Año', 'Mes',
            'Viajes PDF', 'Ingreso Bruto PDF', 'Horas PDF',
            'Propinas PDF', 'Incentivos PDF', 'Impuestos PDF',
            'Viajes RED', 'Ingreso Bruto RED', 'Duración (min)', 'Distancia (km)',
            'Δ Viajes', 'Δ Ingreso (MXN)', '% Δ Ingreso',
            'Efic. MXN/min', 'Efic. MXN/km',
        ]
        _header_row(ws, headers, row=5, bg=C['header'])

        fmt = {
            3: '$#,##0.00', 4: '0.00', 5: '$#,##0.00', 6: '$#,##0.00', 7: '$#,##0.00',
            9: '$#,##0.00', 10: '0.00', 11: '0.00',
            13: '$#,##0.00', 14: '0.00%', 15: '$#,##0.000', 16: '$#,##0.000'
        }

        for i, row in df_audit.iterrows():
            r  = i + 6
            bg = C['gris_claro'] if i % 2 == 0 else C['white']
            pct = row['pct_delta'] / 100 if pd.notna(row['pct_delta']) else 0
            vals = [
                row['anio'], row['mes'],
                row['viajes_pdf'], row['ingreso_bruto_pdf'], row['horas_pdf'],
                row['propinas_pdf'], row['incentivos_pdf'], row['impuestos_pdf'],
                row['viajes_red'], row['ingreso_bruto_red'],
                row['duracion_min_red'], row['distancia_km_red'],
                row['delta_viajes'], row['delta_ingreso'], pct,
                row['eficiencia_mxn_min'], row.get('eficiencia_mxn_km', ''),
            ]
            _data_row(ws, r, vals, bg=bg, fmt_map=fmt)

            # Color-coding de deltas: verde = cuadra, rojo = discrepancia
            for col_i, field in [(13, 'delta_viajes'), (14, 'delta_ingreso')]:
                c = ws.cell(r, col_i)
                c.fill = PatternFill('solid',
                    fgColor=C['green_fill'] if abs(row[field]) <= 0.5 else C['red_fill'])

        # Fila de totales
        tr = len(df_audit) + 6
        for col in range(1, 18):
            ws.cell(tr, col).fill   = PatternFill('solid', fgColor=C['coral_dark'])
            ws.cell(tr, col).border = _borde()
        ws.cell(tr, 1, 'TOTAL').font = Font(name='Arial', bold=True, size=10, color=C['white'])
        ws.cell(tr, 1).fill = PatternFill('solid', fgColor=C['coral_dark'])
        for ci, cl, nf in [
            (3,'C','0'), (4,'D','$#,##0.00'), (5,'E','0.00'),
            (6,'F','$#,##0.00'), (7,'G','$#,##0.00'), (8,'H','$#,##0.00'),
            (9,'I','0'), (10,'J','$#,##0.00'), (11,'K','0.00'), (12,'L','0.00'),
            (13,'M','0'), (14,'N','$#,##0.00'),
        ]:
            c = ws.cell(tr, ci, f'=SUM({cl}6:{cl}{tr-1})')
            c.font   = Font(name='Arial', bold=True, size=10, color=C['white'])
            c.fill   = PatternFill('solid', fgColor=C['coral_dark'])
            c.border = _borde()
            c.number_format = nf
        # Promedio % delta
        pc = ws.cell(tr, 15, f'=AVERAGE(O6:O{tr-1})')
        pc.font   = Font(name='Arial', bold=True, size=10, color=C['white'])
        pc.fill   = PatternFill('solid', fgColor=C['coral_dark'])
        pc.border = _borde()
        pc.number_format = '0.00%'

        ws.freeze_panes = 'C6'   # Freeze año+mes al hacer scroll horizontal
        _col_widths(ws, [6, 13, 11, 19, 10, 14, 14, 14, 11, 19, 14, 13, 11, 19, 12, 14, 13])

    # ── Método principal ──────────────────────────────────────────────────────
    def ejecutar_y_exportar(self, output_file: str = "YAGA_DataSet_Final.xlsx"):
        print("=" * 60)
        print("⚖️  YAGA DATA FUSION v8.0 — PDF vs RED")
        print("=" * 60)

        # Carga
        df_json = self._cargar_json()
        df_pdf  = self._cargar_pdf()

        # Agregaciones mensuales
        df_red_agg = (
            df_json.groupby(['anio', 'mes_texto'])
            .agg(viajes_red=('trip_id','count'), ingreso_bruto_red=('monto_bruto','sum'),
                 duracion_min_red=('duracion_min','sum'), distancia_km_red=('distancia_km','sum'))
            .reset_index().rename(columns={'mes_texto': 'mes'})
        )
        df_pdf_agg = (
            df_pdf.groupby(['anio', 'mes'])
            .agg(viajes_pdf=('Viajes_Totales','sum'), ingreso_bruto_pdf=('Monto_Bruto_Generado','sum'),
                 horas_pdf=('Horas_Conectado','sum'), propinas_pdf=('Propinas','sum'),
                 incentivos_pdf=('Incentivos','sum'), impuestos_pdf=('Impuestos','sum'))
            .reset_index()
        )

        # Auditoría
        df_audit = pd.merge(df_pdf_agg, df_red_agg, on=['anio', 'mes'], how='inner')
        df_audit['delta_viajes']      = df_audit['viajes_red']        - df_audit['viajes_pdf']
        df_audit['delta_ingreso']     = df_audit['ingreso_bruto_red'] - df_audit['ingreso_bruto_pdf']
        df_audit['pct_delta']         = (df_audit['delta_ingreso'] / df_audit['ingreso_bruto_pdf'] * 100).round(2)
        df_audit['eficiencia_mxn_min']= (df_audit['ingreso_bruto_red'] / df_audit['duracion_min_red']).round(2)
        df_audit['eficiencia_mxn_km'] = (df_audit['ingreso_bruto_red'] / df_audit['distancia_km_red']).round(2)
        df_audit['_ord'] = df_audit['mes'].map(MESES_ORD)
        df_audit = df_audit.sort_values(['anio', '_ord']).drop(columns='_ord').reset_index(drop=True)

        # Construir XLSX
        wb = Workbook()
        wb.remove(wb.active)
        self._sheet_viajes(wb, df_json)
        self._sheet_pdf(wb, df_pdf)
        self._sheet_auditoria(wb, df_audit)
        wb.save(output_file)

        print(f"✅  {len(df_json):,} viajes · {len(df_pdf):,} semanas · {len(df_audit)} meses auditados")
        print(f"💾  Exportado: {output_file}")
        return df_audit


# ── Ejecución y visualización en notebook ────────────────────────────────────
motor = YagaDataFusionEngine()
df_auditoria = motor.ejecutar_y_exportar()

if df_auditoria is not None:
    fmt_vista = {
        'ingreso_bruto_pdf':  '${:,.2f}',
        'ingreso_bruto_red':  '${:,.2f}',
        'delta_ingreso':      '${:,.2f}',
        'pct_delta':          '{:.2f}%',
        'eficiencia_mxn_min': '${:,.2f}/min',
    }

    def _colorear(val):
        return f'color: {"green" if abs(val) <= 0.5 else "red"}'

    cols_vista = ['anio', 'mes', 'viajes_pdf', 'viajes_red', 'delta_viajes',
                  'ingreso_bruto_pdf', 'ingreso_bruto_red', 'delta_ingreso',
                  'pct_delta', 'eficiencia_mxn_min']

    print("\n📊  VISTA EJECUTIVA:")
    display(
        df_auditoria[cols_vista]
        .style
        .format(fmt_vista)
        .map(_colorear, subset=['delta_viajes', 'delta_ingreso'])
    )

⚖️  YAGA DATA FUSION v8.0 — PDF vs RED
✅  3,436 viajes · 49 semanas · 13 meses auditados
💾  Exportado: YAGA_DataSet_Final.xlsx

📊  VISTA EJECUTIVA:


,anio,mes,viajes_pdf,viajes_red,delta_viajes,ingreso_bruto_pdf,ingreso_bruto_red,delta_ingreso,pct_delta,eficiencia_mxn_min
0,2024,Junio,38,40,2,"$4,509.12","$4,246.29",$-262.83,-5.83%,$4.03/min
1,2024,Julio,18,23,5,"$1,512.86","$1,836.30",$323.44,21.38%,$4.86/min
2,2024,Diciembre,368,279,-89,"$36,761.48","$29,823.96","$-6,937.52",-18.87%,$5.68/min
3,2025,Enero,472,521,49,"$33,455.92","$35,935.03","$2,479.11",7.41%,$4.42/min
4,2025,Febrero,330,340,10,"$18,142.86","$19,927.27","$1,784.41",9.84%,$4.09/min
5,2025,Marzo,415,393,-22,"$21,382.58","$20,525.32",$-857.26,-4.01%,$4.12/min
6,2025,Abril,315,292,-23,"$15,389.01","$14,652.00",$-737.01,-4.79%,$4.30/min
7,2025,Mayo,277,362,85,"$17,163.78","$20,320.81","$3,157.03",18.39%,$4.57/min
8,2025,Junio,317,284,-33,"$27,083.45","$23,352.29","$-3,731.16",-13.78%,$4.72/min
9,2025,Julio,223,230,7,"$20,394.45","$20,442.14",$47.69,0.23%,$4.58/min


In [5]:
class YagaTravelExtractor:
    """
    Módulo de Extracción Quirúrgica YAGA v8.2
    Objetivo: Convertir únicamente la base de viajes a CSV plano.
    """
    def __init__(self, json_file="YAGA_DataSet_Operativo_ViajesJSON_2024_2026.csv"):
        self.json_file = json_file

    def exportar_viajes_csv(self, output_name="YAGA_Viajes_Individuales_Final.csv"):
        if not os.path.exists(self.json_file):
            print(f"❌ Error: No se encuentra el origen {self.json_file}")
            return

        # Carga y limpieza técnica (Fase: Data Preparation)
        df = pd.read_csv(self.json_file)
        
        # 1. Eliminar duplicados por Trip ID para integridad del modelo [cite: 73, 1331]
        df = df.drop_duplicates(subset=['trip_id'])
        
        # 2. Formateo de fechas para compatibilidad con R (ISO 8601) [cite: 927]
        df['fecha_local'] = pd.to_datetime(df['fecha_local'])
        
        # 3. Exportación limpia (Sin nulos, UTF-8) [cite: 495, 834]
        df.to_csv(output_name, index=False, encoding='utf-8')
        
        print(f"✅ Extracción Exitosa: {len(df):,} viajes exportados.")
        print(f"💾 Archivo generado: {output_name}")
        return df

# Ejecución directa
extractor = YagaTravelExtractor()
df_viajes = extractor.exportar_viajes_csv()

✅ Extracción Exitosa: 3,436 viajes exportados.
💾 Archivo generado: YAGA_Viajes_Individuales_Final.csv


In [6]:
# 1. Recuperamos df_maestro desde el archivo generado previamente
# (Asegúrate de que 'YAGA_Viajes_Individuales_Final.csv' esté en tu carpeta)
df_maestro = pd.read_csv('YAGA_Viajes_Individuales_Final.csv')

# 2. Re-aplicamos Limpieza (Task 3.2: Clean Data)
df_maestro_limpio = df_maestro.dropna(subset=['origen', 'destino']).copy()
df_maestro_limpio = df_maestro_limpio[df_maestro_limpio['estado'].fillna('').str.lower() == 'completed']

# 3. Re-aplicamos Clasificación Regional (Task 3.3: Construct Data)
def motor_clasificacion_v2(row):
    texto = f"{row['origen']} {row['destino']}".lower()
    valle_mex_tags = ['cdmx', 'ciudad de méxico', 'mex', 'edo', 'df', 'mexico', 'méxico',
                      'ecatepec', 'nezahualcóyotl', 'texcoco', '特斯科科', '墨西哥城', 
                      'naucalpan', 'chimalhuacán', 'chicoloapan', 'atenco', 'tlalnepantla',
                      'coyoacán', 'cuauhtémoc', 'iztacalco', 'xochimilco', 'obregón']
    morelia_tags = ['morelia', 'michoacán', 'mich']
    
    if any(tag in texto for tag in valle_mex_tags): return 'Valle de México'
    if any(tag in texto for tag in morelia_tags): return 'Morelia'
    return 'Otras / Por Clasificar'

df_maestro_limpio['ciudad'] = df_maestro_limpio.apply(motor_clasificacion_v2, axis=1)

# 4. Re-aplicamos Organización de Columnas (Task 3.5: Format Data)
cols_kpis = ['trip_id', 'fecha_local', 'monto_bruto', 'duracion_min', 'distancia_km', 'estado']
cols_geografia = ['ciudad', 'origen', 'destino']
columnas_finales = [c for c in cols_kpis + cols_geografia if c in df_maestro_limpio.columns]

# RESTABLECIDO: Aquí nace de nuevo df_maestro_final
df_maestro_final = df_maestro_limpio[columnas_finales].copy()

print(f"🛡️ YAGA Core: 'df_maestro_final' ha sido restaurado con {len(df_maestro_final)} registros.")

🛡️ YAGA Core: 'df_maestro_final' ha sido restaurado con 3300 registros.


In [7]:
# YAGA Project - Data Sanitization Module
# Fase: Data Preparation | Tarea: Clean Data

# 1. Identificar el tamaño original para el Data Cleaning Report
total_original = len(df_maestro)

# 2. Purgar registros nulos en columnas logísticas clave y viajes no completados
# Eliminamos filas donde 'origen' o 'destino' sean nulos
df_maestro_limpio = df_maestro.dropna(subset=['origen', 'destino']).copy()

# 3. Filtrar estrictamente por estado 'completed' (si la columna existe y no es nula)
# Usamos .fillna('') para evitar errores al comparar strings con NaN
df_maestro_limpio = df_maestro_limpio[df_maestro_limpio['estado'].fillna('').str.lower() == 'completed']

# 4. Cuantificar el impacto de la limpieza
total_limpio = len(df_maestro_limpio)
registros_eliminados = total_original - total_limpio

print("🛡️ YAGA Audit: Protocolo de Limpieza Ejecutado")
print(f"Registros originales: {total_original}")
print(f"Registros conservados: {total_limpio}")
print(f"Ajustes/Nulos eliminados: {registros_eliminados}")

🛡️ YAGA Audit: Protocolo de Limpieza Ejecutado
Registros originales: 3436
Registros conservados: 3300
Ajustes/Nulos eliminados: 136


In [8]:
# YAGA Project - Final Regional Consolidation
# Phase: Data Preparation | Task: Construct Data

def motor_clasificacion_v2(row):
    texto = f"{row['origen']} {row['destino']}".lower()
    
    # Diccionario de etiquetas detectadas en la auditoría
    # Incluimos términos en caracteres chinos detectados en el tráfico de red
    valle_mexico_tags = [
        'cdmx', 'ciudad de méxico', 'mex', 'edo', 'df', 'mexico', 'méxico',
        'ecatepec', 'nezahualcóyotl', 'texcoco', '特斯科科', '墨西哥城', 
        'naucalpan', 'chimalhuacán', 'chicoloapan', 'atenco', 'tlalnepantla',
        'coyoacán', 'cuauhtémoc', 'iztacalco', 'xochimilco', 'obregón'
    ]
    
    morelia_tags = ['morelia', 'michoacán', 'mich']
    
    if any(tag in texto for tag in valle_mexico_tags):
        return 'Valle de México'
    elif any(tag in texto for tag in morelia_tags):
        return 'Morelia'
    else:
        return 'Otras / Por Clasificar'

# Aplicamos la corrección final al DataSet Maestro
df_maestro_limpio['ciudad'] = df_maestro_limpio.apply(motor_clasificacion_v2, axis=1)

# Verificación de cierre (Task 2.4: Verify Data Quality)
print("📊 Consolidación Final de Plazas:")
print(df_maestro_limpio['ciudad'].value_counts())

📊 Consolidación Final de Plazas:
ciudad
Valle de México    1964
Morelia            1336
Name: count, dtype: int64


In [10]:
import pandas as pd
import numpy as np

# 1. Cálculo de Eficiencia Logística (MXN/KM)
# Usamos replace y fillna para manejar divisiones por cero o nulos
df_maestro_final['eficiencia_por_km'] = (
    df_maestro_final['monto_bruto'] / df_maestro_final['distancia_km']
).replace([np.inf, -np.inf], 0).fillna(0)

# Redondeamos a 2 decimales por estándar financiero
df_maestro_final['eficiencia_por_km'] = df_maestro_final['eficiencia_por_km'].round(2)

# 2. Mantenimiento del Orden Arquitectónico (Task 3.5: Format Data)
# Aseguramos que los datos de ubicación sigan al final por Privacidad
cols_geografia = ['ciudad', 'origen', 'destino']
cols_kpis = [c for c in df_maestro_final.columns if c not in cols_geografia]

df_maestro_final = df_maestro_final[cols_kpis + cols_geografia]

# 3. Auditoría Visual Rápida (Task 2.4: Verify Data Quality)
print("🚀 YAGA Core: KPI 'Eficiencia por KM' inyectado exitosamente.")
display(df_maestro_final[['trip_id', 'monto_bruto', 'distancia_km', 'eficiencia_por_km', 'ciudad']].head(10))

🚀 YAGA Core: KPI 'Eficiencia por KM' inyectado exitosamente.


,trip_id,monto_bruto,distancia_km,eficiencia_por_km,ciudad
1,bfb75c18-00a0-4c6c-8bc7-5c0dfbc73d68,137.55,18.75,7.34,Valle de México
2,9c21a397-1b0b-41d8-bc34-5cc335570c74,146.47,26.43,5.54,Valle de México
3,b22d954c-e85b-4373-af7c-cd4a9b0fd492,90.52,6.51,13.90,Valle de México
4,7817b52a-1257-4b8f-9f1f-363a762cdc59,189.40,30.02,6.31,Valle de México
5,9a5b45fd-0dfe-4cc3-8130-f33dfdc22711,72.58,4.29,16.92,Valle de México
6,804f49ac-b0cc-46c9-9c1b-5caac2e567de,91.95,14.23,6.46,Valle de México
7,e856cbcd-8824-4de3-bbed-3d775e1fbbf8,90.86,14.40,6.31,Valle de México
8,71464039-cfe4-4764-91cb-f812e67d89ae,125.80,14.68,8.57,Valle de México
9,dff2da7d-9a01-475e-ac52-a65e59afe638,71.42,5.36,13.32,Valle de México
10,26407f3f-f4e8-4dba-8663-bbc4cf98d3be,75.89,11.97,6.34,Valle de México


In [11]:
# YAGA Project - Final Architectural Reordering
# Phase: Data Preparation | Task: 3.5 Format Data

# 1. Definir el orden lógico definitivo
# Estructura: Identificadores -> Tiempo -> KPIs Financieros -> Estado -> Geografía
orden_final = [
    'trip_id', 'fecha_local', 'monto_bruto', 
    'duracion_min', 'distancia_km', 'estado', 
    'eficiencia_por_km', 'ciudad', 'origen', 'destino'
]

# 2. Aplicar el reordenamiento (filtrando solo las columnas que existen)
columnas_disponibles = [c for c in orden_final if c in df_maestro_final.columns]
df_maestro_final = df_maestro_final[columnas_disponibles].copy()

# 3. Verificación de salida
print("🛡️ Estructura YAGA consolidada con éxito.")
display(df_maestro_final.head(10))

🛡️ Estructura YAGA consolidada con éxito.


,trip_id,fecha_local,monto_bruto,duracion_min,distancia_km,estado,eficiencia_por_km,ciudad,origen,destino
1,bfb75c18-00a0-4c6c-8bc7-5c0dfbc73d68,2024-06-22 14:40:25,137.55,36.47,18.75,COMPLETED,7.34,Valle de México,"Calle de la Lámpara, 54783 Teoloyucan, México, MX","C. Nayarit, Cuautitlán Izcalli, 54753 Estado d..."
2,9c21a397-1b0b-41d8-bc34-5cc335570c74,2024-06-22 15:21:42,146.47,41.42,26.43,COMPLETED,5.54,Valle de México,"Hacienda Sierra Vieja Lt 2, Fracc Hacienda del...","Av Paseo de la Reforma, Col Juárez Cuauhtémoc,..."
3,b22d954c-e85b-4373-af7c-cd4a9b0fd492,2024-06-22 16:21:45,90.52,17.22,6.51,COMPLETED,13.90,Valle de México,"Col Tabacalera, 06030 Cuauhtémoc, CMX, MX","Eje Central Lázaro Cárdenas, México D.F., 0776..."
4,7817b52a-1257-4b8f-9f1f-363a762cdc59,2024-06-22 16:45:58,189.40,57.92,30.02,COMPLETED,6.31,Valle de México,"Col Maximino Ávila Camacho, 07380 Gustavo A. M...","Niño Artillero, México D.F., 09180 Ciudad de M..."
5,9a5b45fd-0dfe-4cc3-8130-f33dfdc22711,2024-06-22 18:16:44,72.58,9.43,4.29,COMPLETED,16.92,Valle de México,"Congreso de Chilpancingo Nte y Sur, Col Ermita...","09140 Iztapalapa, Ciudad de México, MX"
6,804f49ac-b0cc-46c9-9c1b-5caac2e567de,2024-06-22 18:35:33,91.95,32.17,14.23,COMPLETED,6.46,Valle de México,"Calle Tizapán, 57760 Ciudad Nezahualcóyotl, Mé...","Infonavit Iztacalco, 08900 Iztacalco, Ciudad d..."
7,e856cbcd-8824-4de3-bbed-3d775e1fbbf8,2024-06-22 19:15:49,90.86,24.95,14.40,COMPLETED,6.31,Valle de México,"Ampliación Ramos Millán, 08000 Iztacalco, Ciud...","Ajusco, 04300 Coyoacán, Ciudad de México, MX"
8,71464039-cfe4-4764-91cb-f812e67d89ae,2024-06-22 19:34:30,125.80,33.95,14.68,COMPLETED,8.57,Valle de México,"Los Reyes, 04330 Coyoacán, Ciudad de México, MX","Piloto Adolfo López Mateos, 01290 Álvaro Obreg..."
9,dff2da7d-9a01-475e-ac52-a65e59afe638,2024-06-22 20:32:28,71.42,15.53,5.36,COMPLETED,13.32,Valle de México,"Presidentes Segunda Ampl, 01299 Álvaro Obregón...","Col Merced Gómez, 01480 Álvaro Obregón, DF, MX"
10,26407f3f-f4e8-4dba-8663-bbc4cf98d3be,2024-06-22 20:45:21,75.89,18.35,11.97,COMPLETED,6.34,Valle de México,"Colonia Las Águilas, 01710 Álvaro Obregón, Ciu...","Col Reforma Social Miguel Hidalgo, 11650 Migue..."


In [13]:
# YAGA Project - Master Dataset Persistence
# Phase: Data Preparation | Task: Save Clean Dataset

# 1. Definir el nombre del archivo de salida
nombre_archivo = "YAGA_DataSet_Clean.csv"

# 2. Guardar el DataFrame con la columna 'eficiencia_por_km' ya integrada
df_maestro_final.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')

# 3. Confirmación de guardado
print("="*60)
print("🛡️ YAGA CORE: DATASET LIMPIO Y GUARDADO")
print("="*60)
print(f"✅ Archivo generado: {nombre_archivo}")
print(f"📊 Total de registros validados: {len(df_maestro_final):,}")
print(f"💡 KPI 'eficiencia_por_km' incluido exitosamente.")
print("-" * 60)

🛡️ YAGA CORE: DATASET LIMPIO Y GUARDADO
✅ Archivo generado: YAGA_DataSet_Clean.csv
📊 Total de registros validados: 3,300
💡 KPI 'eficiencia_por_km' incluido exitosamente.
------------------------------------------------------------
